# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** EU General Data Protection Regulation (GDPR), Regulation (EU) 2016/679, official 88-page PDF.

**Models**
| Role | Model |
|---|---|
| Bi-encoder (dense retrieval) | `sentence-transformers/all-MiniLM-L6-v2` |
| Cross-encoder (re-ranker) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Vector store | FAISS `IndexFlatIP` on L2-normalised vectors (inner product = cosine similarity) |
| LLM (answer generation) | Google Gemini `gemini-2.5-flash`, temperature 0, thinking disabled |

**Pipelines (same chunk corpus for both)**
- **Pipeline 1 – Baseline:** bi-encoder → top 10 by cosine similarity → top 3 → LLM → *Answer 1*
- **Pipeline 2 – Re-rank:** same top 10 → cross-encoder scores all 10 (query, chunk) pairs → re-sort → top 3 → LLM (identical prompt) → *Answer 2*

## 1. Setup

In [1]:
# Install once (uncomment if needed):
# !pip install pdfminer.six sentence-transformers faiss-cpu google-genai pandas python-dotenv tabulate

import os, re, time, json, hashlib, statistics
import numpy as np
import pandas as pd
from pdfminer.high_level import extract_pages          # PDF text extraction with coordinates
from pdfminer.layout import LTTextContainer, LTTextLine, LTChar
import faiss                     # vector index
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()                    # reads GEMINI_API_KEY from a local .env file (not committed)
pd.set_option("display.max_colwidth", 120)

PDF_PATH        = "GDPR.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL       = "gemini-2.5-flash"
TOP_K_RETRIEVE  = 10             # candidates from the bi-encoder
TOP_K_CONTEXT   = 3              # chunks handed to the LLM

/Users/kushalkaran/Desktop/RAG-LMA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1019 22:30:46.541000 31207 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Load and clean the PDF

The Official Journal layout causes three problems for naive text extraction:
1. Recital numbers `(4)` and paragraph numbers `1.` sit in a separate left-hand column, so default extraction detaches them from their text.
   → Text lines are rebuilt from their **coordinates**: every fragment on the same baseline is joined left-to-right.
2. Footnotes citing other legal acts (`(1) OJ C 229, ...`) re-use the same `(n)` numbering as recitals.
   → They are printed in a smaller font (~8.5pt vs 9.6pt body), so lines **below 9pt are dropped**.
3. A running header on every page (`L 119/x · EN · Official Journal of the European Union · 4.5.2016`) → removed.

In [2]:
def pdf_lines(path, min_font=9.0):
    # Yield clean text lines in reading order, with footnotes (small font) removed.
    for page in extract_pages(path):
        items = []
        for box in page:
            if not isinstance(box, LTTextContainer):
                continue
            sizes = [ch.size for line in box if isinstance(line, LTTextLine) for ch in line if isinstance(ch, LTChar)]
            # footnote block = small font AND starts with a "(n)" marker (small-caps headings are also small, keep those)
            if sizes and sum(sizes) / len(sizes) < min_font and re.match(r"\(\d+\)", box.get_text().strip()):
                continue
            for line in box:
                if isinstance(line, LTTextLine) and line.get_text().strip():
                    items.append((round(line.y0), line.x0, line.get_text().strip()))
        items.sort(key=lambda t: (-t[0], t[1]))          # top-to-bottom, then left-to-right
        row, row_y = [], None
        for y, x, txt in items:
            if row_y is not None and abs(y - row_y) > 2:  # new baseline -> flush the previous row
                yield " ".join(t for _, t in sorted(row)); row = []
            row.append((x, txt)); row_y = y
        if row:
            yield " ".join(t for _, t in sorted(row))

HEADER_LINE = re.compile(r"Official Journal of the European Union|^L 119/\d+$|^\d{1,2}\.\d{1,2}\.\d{4}$")

t0 = time.perf_counter()
lines = [re.sub(r"\s+", " ", l) for l in pdf_lines(PDF_PATH)]
lines = [l for l in lines if not HEADER_LINE.search(l)]
raw = "\n" + "\n".join(lines)
print(f"{len(lines):,} lines extracted in {time.perf_counter()-t0:.1f}s")

# Split the preamble (recitals) from the enacting terms (articles).
split_at = raw.index("HAVE ADOPTED THIS REGULATION")
recital_text, article_text = raw[:split_at], raw[split_at:]
print(f"recital section: {len(recital_text):,} chars | article section: {len(article_text):,} chars")

3,673 lines extracted in 5.6s
recital section: 154,824 chars | article section: 191,594 chars


## 3. Structure-aware chunking

Instead of fixed-size windows, chunks follow the legal structure so that **a rule and its exceptions land in different chunks**
(e.g. Art. 9(1) prohibition vs. Art. 9(2) exceptions). This is what makes the failure modes observable.

- **Recitals** → one chunk per recital `(1)…(173)`. A few long recitals exceed the bi-encoder's 256-token window and are truncated
  when embedded (the cross-encoder reads up to 512 tokens). Splitting them was tested but produced more distractor chunks and worse article recall.
- **Articles** → one chunk per numbered paragraph (`Art. 17(3)`), prefixed with the article number and title so each chunk is self-describing.
- Paragraphs longer than ~1,200 characters are split further on their points `(a)`, `(b)`, … or `(1)`, `(2)`, … (keeping the paragraph lead-in).
  This keeps chunks within the bi-encoder's 256-token window, which would otherwise silently truncate long chunks.

In [3]:
def norm(s):
    s = s.replace("­", "")                      # soft hyphens
    s = re.sub(r"-\n(?=[a-z])", "", s)               # words hyphenated across lines
    return re.sub(r"\s+", " ", s).strip()

MAX_CHARS = 1200

def split_long(prefix, body):
    # Split a long paragraph on its (a), (b)... points, repeating the lead-in sentence for context.
    if len(body) <= MAX_CHARS:
        return [body]
    parts = re.split(r"\s(?=\((?:[a-z]{1,2}|\d{1,2})\)\s)", body)   # (a) points, or (1) definitions in Art. 4
    lead, points = parts[0], parts[1:]
    if not points:                                     # no points (e.g. long recitals) -> split on sentences
        out, cur = [], ""
        for sent in re.split(r"(?<=[.;])\s+(?=[A-Z])", body):
            if cur and len(cur) + len(sent) > MAX_CHARS:
                out.append(cur); cur = sent
            else:
                cur = (cur + " " + sent).strip()
        return out + [cur]
    out, cur = [], lead
    for p in points:
        if len(cur) + len(p) > MAX_CHARS and cur != lead:
            out.append(cur)
            cur = lead + " … " + p
        else:
            cur += " " + p
    out.append(cur)
    return out

chunks = []   # each: dict(id, kind, text)

# --- recitals: "(n) text" with n increasing 1..173 -------------------------
expected = 1
pos = 0
recital_starts = []
for m in re.finditer(r"(?:^|\n)\((\d{1,3})\)\s", recital_text):
    if int(m.group(1)) == expected:
        recital_starts.append((expected, m.start(), m.end()))
        expected += 1
for i, (n, s, e) in enumerate(recital_starts):
    end = recital_starts[i + 1][1] if i + 1 < len(recital_starts) else len(recital_text)
    # Recitals are kept whole: they are explanatory (non-binding) text, and splitting them multiplies
    # near-duplicate distractor chunks that crowd the binding articles out of the top 10.
    chunks.append(dict(id=f"Recital {n}", kind="recital",
                       text=f"Recital ({n}): " + norm(recital_text[e:end])))

# --- articles: "Article n\nTitle\n1. ... 2. ..." ---------------------------
article_text = re.sub(r"\nCHAPTER [IVX]+\n[^\n]+(?:\n[^\n]+)?(?=\nArticle|\nSection)", "\n", article_text)
article_text = re.sub(r"\nSection \d+\n[^\n]+(?=\nArticle)", "\n", article_text)
art_iter, want = [], 1
for m in re.finditer(r"\nArticle (\d{1,2})\n([^\n]+(?:\n[a-z][^\n]*)*)\n", article_text):   # titles may wrap onto lines starting lower-case
    if int(m.group(1)) == want:                       # articles must appear in order 1..99
        art_iter.append(m); want += 1
for i, m in enumerate(art_iter):
    num, title = int(m.group(1)), norm(m.group(2))
    end = art_iter[i + 1].start() if i + 1 < len(art_iter) else len(article_text)
    body = article_text[m.end():end]
    # numbered paragraphs "1. ", "2. " ... must appear in order at line starts
    starts, want = [], 1
    for pm in re.finditer(r"(?:^|\n)(\d{1,2})\.\s", body):
        if int(pm.group(1)) == want:
            starts.append((want, pm.start(), pm.end())); want += 1
    if not starts:
        paras = [(None, norm(body))]
    else:
        paras = [(n, norm(body[e:(starts[j + 1][1] if j + 1 < len(starts) else len(body))]))
                 for j, (n, s, e) in enumerate(starts)]
    for n, ptxt in paras:
        cid = f"Art. {num}" + (f"({n})" if n else "")
        prefix = f"Article {num} – {title}" + (f", paragraph {n}" if n else "") + ": "
        pieces = split_long(prefix, ptxt)
        for k, piece in enumerate(pieces):
            chunks.append(dict(id=cid + (f" [part {k+1}]" if len(pieces) > 1 else ""),
                               kind="article", text=prefix + piece))

chunk_df = pd.DataFrame(chunks)
chunk_df["chars"] = chunk_df.text.str.len()
print(f"{len(chunk_df)} chunks  |  recitals: {(chunk_df.kind=='recital').sum()}  "
      f"|  article chunks: {(chunk_df.kind=='article').sum()}  "
      f"|  articles found: {chunk_df[chunk_df.kind=='article'].id.str.extract(r'Art\. (\d+)')[0].nunique()}")
print(chunk_df.chars.describe().round(0).to_dict())
chunk_df[chunk_df.id.str.startswith("Art. 9(")][["id", "text"]]

598 chunks  |  recitals: 173  |  article chunks: 425  |  articles found: 99
{'count': 598.0, 'mean': 625.0, 'std': 446.0, 'min': 102.0, '25%': 308.0, '50%': 466.0, '75%': 834.0, 'max': 3075.0}


,id,text
208,Art. 9(1),"Article 9 – Processing of special categories of personal data, paragraph 1: Processing of personal data revealing ra..."
209,Art. 9(2) [part 1],"Article 9 – Processing of special categories of personal data, paragraph 2: Paragraph 1 shall not apply if one of th..."
210,Art. 9(2) [part 2],"Article 9 – Processing of special categories of personal data, paragraph 2: Paragraph 1 shall not apply if one of th..."
211,Art. 9(2) [part 3],"Article 9 – Processing of special categories of personal data, paragraph 2: Paragraph 1 shall not apply if one of th..."
212,Art. 9(2) [part 4],"Article 9 – Processing of special categories of personal data, paragraph 2: Paragraph 1 shall not apply if one of th..."
213,Art. 9(3),"Article 9 – Processing of special categories of personal data, paragraph 3: Personal data referred to in paragraph 1..."
214,Art. 9(4),"Article 9 – Processing of special categories of personal data, paragraph 4: Member States may maintain or introduce ..."


## 4. Pipeline 1 – Bi-encoder index + baseline retrieval

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER)
cross_encoder = CrossEncoder(CROSS_ENCODER)

texts = chunk_df.text.tolist()
ids = chunk_df.id.tolist()

t0 = time.perf_counter()
emb = bi_encoder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(emb.shape[1])     # inner product on unit vectors == cosine similarity
index.add(emb.astype("float32"))
print(f"Indexed {index.ntotal} chunks, dim={emb.shape[1]}, one-off indexing time {time.perf_counter()-t0:.1f}s")

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Embed the query independently and return the top-k chunks by cosine similarity.
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, idx = index.search(q.astype("float32"), k)
    return [dict(chunk=int(i), id=ids[i], bi_score=float(s), bi_rank=r + 1)
            for r, (i, s) in enumerate(zip(idx[0], scores[0]))]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5313.10it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6298.74it/s]

Indexed 598 chunks, dim=384, one-off indexing time 5.6s


## 5. Pipeline 2 – Cross-encoder re-ranking of the same top 10

In [5]:
def rerank(query, candidates):
    # Score every (query, chunk) pair jointly with the cross-encoder and re-sort descending.
    pairs = [(query, texts[c["chunk"]]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    out.sort(key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

## 6. LLM generation (identical prompt for both pipelines)

The prompt forces the model to answer **only** from the supplied context. GDPR is well known to LLMs, so without this
restriction the model could answer from memory and hide retrieval failures.

In [6]:
PROMPT = '''You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context passages below. Do not use outside knowledge.
Cite the article or recital number for every claim. If the passages do not contain the answer,
reply exactly: "The provided context does not contain enough information to answer this."
Keep the answer under 120 words.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''

# Every LLM response is cached on disk (keyed by model + full prompt), so re-running the notebook
# reproduces the same answers without spending API quota. The recorded latency is the original call's.
CACHE_PATH = "results/llm_cache.json"
os.makedirs("results", exist_ok=True)
try:
    llm_cache = json.load(open(CACHE_PATH))
except FileNotFoundError:
    llm_cache = {}

def generate(question, top_chunks):
    context = "\n\n".join(f"[{c['id']}] {texts[c['chunk']]}" for c in top_chunks)
    prompt = PROMPT.format(context=context, question=question)
    key = hashlib.sha256(f"{LLM_MODEL}\n{prompt}".encode()).hexdigest()
    if key in llm_cache:
        hit = llm_cache[key]
        return hit["answer"], hit["latency_s"]
    if not os.environ.get("GEMINI_API_KEY"):
        print("  no cached answer and no GEMINI_API_KEY set – skipping generation")
        return "[generation skipped: no API key]", None
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    for attempt in range(4):                      # free-tier per-minute limits -> simple back-off
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)))
            answer, latency = resp.text.strip(), time.perf_counter() - t0
            llm_cache[key] = dict(model=LLM_MODEL, question=question,
                                  chunks=[c["id"] for c in top_chunks], answer=answer, latency_s=latency)
            json.dump(llm_cache, open(CACHE_PATH, "w"), indent=1, ensure_ascii=False)
            return answer, latency
        except Exception as e:
            if "PerDay" in str(e) or "per day" in str(e).lower():   # daily quota gone: retrying won't help
                print(f"  daily LLM quota exhausted for {LLM_MODEL}")
                break
            wait = 15 * (attempt + 1)
            print(f"  LLM error ({type(e).__name__}): retrying in {wait}s")
            time.sleep(wait)
    return "[generation failed]", None

## 7. Run both pipelines end-to-end

In [7]:
def run_both(query, gold=None, show=True):
    # Pipeline 1
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)
    t_retrieve = time.perf_counter() - t0
    base_top3 = cands[:TOP_K_CONTEXT]

    # Pipeline 2 (re-uses the SAME 10 candidates)
    t0 = time.perf_counter()
    reranked = rerank(query, cands)
    t_rerank = time.perf_counter() - t0
    rr_top3 = reranked[:TOP_K_CONTEXT]

    ans1, t_llm1 = generate(query, base_top3)
    ans2, t_llm2 = generate(query, rr_top3)

    table = pd.DataFrame(reranked).sort_values("bi_rank")[["bi_rank", "ce_rank", "id", "bi_score", "ce_score"]]
    table["shift"] = table.bi_rank - table.ce_rank          # positive = promoted by the cross-encoder
    if gold:
        table["gold"] = table.id.apply(lambda i: "★" if any(i.startswith(g) for g in gold) else "")
    res = dict(query=query, gold=gold, table=table, base_top3=[c["id"] for c in base_top3],
               rr_top3=[c["id"] for c in rr_top3], answer1=ans1, answer2=ans2,
               t_retrieve=t_retrieve, t_rerank=t_rerank, t_llm1=t_llm1, t_llm2=t_llm2)
    if show:
        show_result(res)
    return res

def show_result(res):
    from IPython.display import display, Markdown
    display(Markdown(f"### Q: {res['query']}\n*Gold chunk(s): {', '.join(res['gold']) if res['gold'] else '–'}*"))
    display(res["table"].style.hide(axis="index").format({"bi_score": "{:.3f}", "ce_score": "{:.2f}"})
            .apply(lambda r: ["background-color: rgba(255,200,0,.25)" if r.get("gold") == "★" else "" for _ in r], axis=1))
    display(Markdown(f"**Baseline top 3:** {', '.join(res['base_top3'])}  \n"
                     f"**Re-ranked top 3:** {', '.join(res['rr_top3'])}\n\n"
                     f"**Answer 1 (baseline):**\n> {res['answer1'].replace(chr(10), chr(10)+'> ')}\n\n"
                     f"**Answer 2 (re-ranked):**\n> {res['answer2'].replace(chr(10), chr(10)+'> ')}"))

## 8. Failure-mode stress tests

Each query has a hand-labelled **gold** chunk (★) – the passage a correct answer needs, checked against the regulation text.
Gold chunks are highlighted in the rank tables; `shift` = baseline rank − re-ranked rank.

In [8]:
QUERIES = [
 {
  "key": "advertising",
  "label": "Q1 · Keyword trap / synonym – \"advertising\" vs. \"direct marketing\"",
  "why": "The user says *advertising*; the regulation only ever says *direct marketing*. Gold: **Art. 21(3)** – once the data subject objects to direct marketing, the data must no longer be processed for it (an absolute right, unlike the balancing test in Art. 21(1)).",
  "query": "Can I stop a company from using my data for advertising?",
  "gold": [
   "Art. 21(2)",
   "Art. 21(3)"
  ]
 },
 {
  "key": "religion_public",
  "label": "Q2 · Exception / corner case – special-category data the person made public",
  "why": "Religious beliefs are special-category data whose processing is prohibited by Art. 9(1). The relevant exception is buried in a long list: **Art. 9(2)(e)** – data *manifestly made public by the data subject*.",
  "query": "Can someone's religious beliefs be processed if they made them public themselves?",
  "gold": [
   "Art. 9(2)"
  ]
 },
 {
  "key": "health_not_prohibited",
  "label": "Q3 · Negation – \"not prohibited\"",
  "why": "The query is phrased negatively. The bi-encoder matches the topic (health data) but not the polarity, pulling in recitals and the prohibition. Gold: the exception list **Art. 9(2)** (e.g. points (h)/(i) – medicine, public health).",
  "query": "When is it not prohibited to process data about someone's health?",
  "gold": [
   "Art. 9(2)"
  ]
 },
 {
  "key": "refuse_request",
  "label": "Q4 · Exception / role limit – when a controller may refuse a data subject",
  "why": "The general rules on handling requests (Art. 12(2)–(4)) say the controller *shall* act. The exception is **Art. 12(5)(b)**: requests that are *manifestly unfounded or excessive* may be refused.",
  "query": "Can a controller refuse to act on a request from a data subject?",
  "gold": [
   "Art. 12(5)"
  ]
 },
 {
  "key": "fee_copy",
  "label": "Q5 · Exception – \"free of charge … unless\"",
  "why": "Art. 15(3) says the first copy is free and further copies may cost a reasonable fee. The *other* fee rule – charging for manifestly unfounded or excessive requests – is in a different article, **Art. 12(5)(a)**.",
  "query": "Can a company charge me money for giving me a copy of my data?",
  "gold": [
   "Art. 15(3)",
   "Art. 12(5)"
  ]
 },
 {
  "key": "transfer_no_adequacy",
  "label": "Q6 · Buried chunk – international transfers without an adequacy decision",
  "why": "Many recitals talk about transfers and adequacy in general terms. The operative answer is **Art. 49(1)** (derogations: explicit consent, contract, vital interests, …) or **Art. 46(1)** (appropriate safeguards).",
  "query": "Can personal data be sent to a country outside the EU that has no adequacy decision?",
  "gold": [
   "Art. 46(1)",
   "Art. 49(1)"
  ]
 },
 {
  "key": "dpo_fine",
  "label": "Q7 · Limitation – keyword trap the re-ranker cannot fix",
  "why": "\"Data protection officer\" matches Arts. 37–39 almost word for word, but the fine is in **Art. 83(4)**, which only says *\"Articles 8, 11, 25 to 39\"*. The gold chunk is not in the bi-encoder's top 10, so the cross-encoder never sees it.",
  "query": "What fine can a company get for failing to appoint a data protection officer?",
  "gold": [
   "Art. 83(4)"
  ]
 },
 {
  "key": "erasure_exceptions",
  "label": "Q8 · Counter-example – re-ranking makes it worse",
  "why": "The bi-encoder already ranks the exception list **Art. 17(3)** first; the cross-encoder prefers the general erasure rules (17(1), 17(2)) and pushes it to #4.",
  "query": "What are the exceptions to the right to erasure?",
  "gold": [
   "Art. 17(3)"
  ]
 }
]

results = {}

### Q1 · Keyword trap / synonym – "advertising" vs. "direct marketing"
The user says *advertising*; the regulation only ever says *direct marketing*. Gold: **Art. 21(3)** – once the data subject objects to direct marketing, the data must no longer be processed for it (an absolute right, unlike the balancing test in Art. 21(1)).

In [9]:
results['advertising'] = run_both('Can I stop a company from using my data for advertising?', gold=['Art. 21(2)', 'Art. 21(3)'])

### Q: Can I stop a company from using my data for advertising?
*Gold chunk(s): Art. 21(2), Art. 21(3)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,2,Recital 70,0.436,-3.94,-1,
2,4,Recital 28,0.436,-9.32,-2,
3,9,Art. 77(1),0.433,-10.88,-6,
4,5,Recital 63,0.428,-9.81,-1,
5,10,Recital 29,0.424,-10.91,-5,
6,1,Art. 21(3),0.423,-3.86,5,★
7,3,Recital 6,0.421,-7.97,4,
8,6,Art. 80(1),0.420,-9.94,2,
9,7,Art. 12(5),0.413,-10.16,2,
10,8,Art. 25(2),0.409,-10.53,2,


**Baseline top 3:** Recital 70, Recital 28, Art. 77(1)  
**Re-ranked top 3:** Art. 21(3), Recital 70, Recital 6

**Answer 1 (baseline):**
> Yes, you can stop a company from using your data for advertising. Data subjects have the right to object to the processing of their personal data for direct marketing, including related profiling, at any time and free of charge (Recital 70). This right must be explicitly brought to the data subject's attention and presented clearly and separately (Recital 70).

**Answer 2 (re-ranked):**
> Yes, you can stop a company from using your data for advertising. Where you object to processing for direct marketing purposes, your personal data shall no longer be processed for such purposes (Art. 21(3)). This right to object should be explicitly brought to your attention and presented clearly and separately from any other information (Recital 70).

### Q2 · Exception / corner case – special-category data the person made public
Religious beliefs are special-category data whose processing is prohibited by Art. 9(1). The relevant exception is buried in a long list: **Art. 9(2)(e)** – data *manifestly made public by the data subject*.

In [10]:
results['religion_public'] = run_both("Can someone's religious beliefs be processed if they made them public themselves?", gold=['Art. 9(2)'])

### Q: Can someone's religious beliefs be processed if they made them public themselves?
*Gold chunk(s): Art. 9(2)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,2,Recital 55,0.467,0.35,-1,
2,5,Art. 91(1),0.428,-5.32,-3,
3,10,Art. 91(2),0.385,-10.62,-7,
4,4,Recital 158,0.340,-4.81,0,
5,8,Recital 31,0.333,-7.08,-3,
6,7,Art. 6(4) [part 2],0.329,-6.68,-1,
7,9,Recital 165,0.328,-10.24,-2,
8,3,Recital 39,0.321,-3.69,5,
9,6,Art. 21(6),0.319,-5.77,3,
10,1,Art. 9(2) [part 2],0.319,0.36,9,★


**Baseline top 3:** Recital 55, Art. 91(1), Art. 91(2)  
**Re-ranked top 3:** Art. 9(2) [part 2], Recital 55, Recital 39

**Answer 1 (baseline):**
> The provided context does not contain enough information to answer this.

**Answer 2 (re-ranked):**
> Yes, someone's religious beliefs can be processed if they have manifestly made them public themselves (Art. 9(2)(e)).

### Q3 · Negation – "not prohibited"
The query is phrased negatively. The bi-encoder matches the topic (health data) but not the polarity, pulling in recitals and the prohibition. Gold: the exception list **Art. 9(2)** (e.g. points (h)/(i) – medicine, public health).

In [11]:
results['health_not_prohibited'] = run_both("When is it not prohibited to process data about someone's health?", gold=['Art. 9(2)'])

### Q: When is it not prohibited to process data about someone's health?
*Gold chunk(s): Art. 9(2)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,4,Recital 63,0.606,0.03,-3,
2,2,Recital 54,0.601,2.21,0,
3,5,Recital 35,0.601,-3.20,-2,
4,7,Recital 159,0.583,-3.42,-3,
5,1,Recital 52,0.575,3.48,4,
6,3,Art. 9(2) [part 3],0.567,0.79,3,★
7,9,Recital 59,0.567,-3.64,-2,
8,8,Art. 4 [part 5],0.565,-3.56,0,
9,10,Recital 62,0.556,-4.38,-1,
10,6,Art. 15(1) [part 1],0.549,-3.26,4,


**Baseline top 3:** Recital 63, Recital 54, Recital 35  
**Re-ranked top 3:** Recital 52, Recital 54, Art. 9(2) [part 3]

**Answer 1 (baseline):**
> Processing data about someone's health is not prohibited when it is necessary for reasons of public interest in public health, even without the data subject's consent (Recital 54). This processing must be subject to suitable measures to protect individuals' rights and freedoms (Recital 54). "Public health" includes all elements related to health status, healthcare needs, and provision (Recital 54). However, such data should not be processed for other purposes by third parties like employers or insurance companies (Recital 54).

**Answer 2 (re-ranked):**
> Processing health data is not prohibited when necessary for preventive or occupational medicine, medical diagnosis, or health/social care provision/management, based on Union/Member State law or contract with a health professional (Art. 9(2)(h)). It's also allowed for public interest reasons in public health, like protecting against serious cross-border threats, provided Union/Member State law includes suitable safeguards (Art. 9(2)(i)). This includes ensuring quality and cost-effectiveness in health insurance claims (Recital 52). Such processing for public health reasons does not require consent (Recital 54).

### Q4 · Exception / role limit – when a controller may refuse a data subject
The general rules on handling requests (Art. 12(2)–(4)) say the controller *shall* act. The exception is **Art. 12(5)(b)**: requests that are *manifestly unfounded or excessive* may be refused.

In [12]:
results['refuse_request'] = run_both('Can a controller refuse to act on a request from a data subject?', gold=['Art. 12(5)'])

### Q: Can a controller refuse to act on a request from a data subject?
*Gold chunk(s): Art. 12(5)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,1,Art. 12(2),0.654,9.54,0,
2,5,Recital 64,0.586,5.15,-3,
3,2,Art. 12(4),0.579,8.00,1,
4,7,Art. 18(1),0.569,4.19,-3,
5,9,Art. 20(1),0.563,1.51,-4,
6,4,Recital 57,0.560,6.72,2,
7,3,Art. 12(5),0.560,7.95,4,★
8,8,Art. 15(1) [part 2],0.557,2.57,0,
9,6,Recital 59,0.551,4.80,3,
10,10,Recital 47,0.548,1.43,0,


**Baseline top 3:** Art. 12(2), Recital 64, Art. 12(4)  
**Re-ranked top 3:** Art. 12(2), Art. 12(4), Art. 12(5)

**Answer 1 (baseline):**
> Yes, a controller can refuse to act on a data subject's request if they demonstrate they cannot identify the data subject (Art. 12(2)). The controller should use all reasonable measures to verify the data subject's identity (Recital 64). If the controller refuses, they must inform the data subject of the reasons and their right to complain to a supervisory authority or seek a judicial remedy within one month (Art. 12(4)).

**Answer 2 (re-ranked):**
> Yes, a controller can refuse to act on a request from a data subject if the requests are manifestly unfounded or excessive, especially due to their repetitive nature (Art. 12(5)). In such cases, the controller must demonstrate the unfounded or excessive character of the request (Art. 12(5)). Additionally, if the controller is not in a position to identify the data subject, they may refuse to act on the request (Art. 12(2)). If a controller refuses to act, they must inform the data subject of the reasons and the possibility of lodging a complaint or seeking a judicial remedy within one month (Art. 12(4)).

### Q5 · Exception – "free of charge … unless"
Art. 15(3) says the first copy is free and further copies may cost a reasonable fee. The *other* fee rule – charging for manifestly unfounded or excessive requests – is in a different article, **Art. 12(5)(a)**.

In [13]:
results['fee_copy'] = run_both('Can a company charge me money for giving me a copy of my data?', gold=['Art. 15(3)', 'Art. 12(5)'])

### Q: Can a company charge me money for giving me a copy of my data?
*Gold chunk(s): Art. 15(3), Art. 12(5)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,1,Art. 15(3),0.588,0.55,0,★
2,3,Recital 6,0.501,-8.46,-1,
3,9,Recital 62,0.472,-9.70,-6,
4,10,Recital 102,0.472,-9.73,-6,
5,8,Recital 142,0.470,-9.66,-3,
6,5,Art. 80(1),0.447,-9.26,1,
7,2,Art. 12(5),0.442,-6.00,5,★
8,6,Recital 63,0.442,-9.46,2,
9,7,Art. 5(1) [part 2],0.439,-9.66,2,
10,4,Recital 31,0.433,-8.91,6,


**Baseline top 3:** Art. 15(3), Recital 6, Recital 62  
**Re-ranked top 3:** Art. 15(3), Art. 12(5), Recital 6

**Answer 1 (baseline):**
> Yes, a company can charge you for copies of your data, but only under specific conditions. The controller must provide the first copy of your personal data free of charge (Art. 15(3)). However, for any further copies requested, the controller may charge a reasonable fee based on administrative costs (Art. 15(3)).

**Answer 2 (re-ranked):**
> Yes, a company can charge you for copies of your data under certain circumstances. The first copy of your personal data must be provided free of charge (Art. 12(5)). However, for any further copies requested, the controller may charge a reasonable fee based on administrative costs (Art. 15(3)). Additionally, if your requests are manifestly unfounded or excessive, especially due to their repetitive nature, the controller may charge a reasonable fee (Art. 12(5)).

### Q6 · Buried chunk – international transfers without an adequacy decision
Many recitals talk about transfers and adequacy in general terms. The operative answer is **Art. 49(1)** (derogations: explicit consent, contract, vital interests, …) or **Art. 46(1)** (appropriate safeguards).

In [14]:
results['transfer_no_adequacy'] = run_both('Can personal data be sent to a country outside the EU that has no adequacy decision?', gold=['Art. 46(1)', 'Art. 49(1)'])

### Q: Can personal data be sent to a country outside the EU that has no adequacy decision?
*Gold chunk(s): Art. 46(1), Art. 49(1)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,4,Recital 102,0.644,1.47,-3,
2,2,Recital 107,0.581,2.51,0,
3,7,Recital 17,0.579,-1.36,-4,
4,1,Recital 101,0.575,4.31,3,
5,8,Art. 50,0.573,-3.60,-3,
6,9,Recital 6,0.570,-3.74,-3,
7,10,Art. 2(3),0.556,-4.56,-3,
8,6,Recital 112,0.549,0.49,2,
9,3,Art. 49(1) [part 2],0.541,1.71,6,★
10,5,Art. 15(2),0.536,0.99,5,


**Baseline top 3:** Recital 102, Recital 107, Recital 17  
**Re-ranked top 3:** Recital 101, Recital 107, Art. 49(1) [part 2]

**Answer 1 (baseline):**
> Yes, personal data can be sent to a country outside the EU that has no adequacy decision. However, this is only permissible if the requirements in this Regulation relating to transfers subject to appropriate safeguards, including binding corporate rules, and derogations for specific situations are fulfilled (Recital 107). Member States may also conclude international agreements for such transfers, provided they do not affect this Regulation or other Union law and include appropriate protection for fundamental rights (Recital 102).

**Answer 2 (re-ranked):**
> Yes, personal data can be sent to a country outside the EU without an adequacy decision (Art. 49(1)). This can occur if appropriate safeguards are in place, such as binding corporate rules (Recital 107, Art. 49(1)). Additionally, transfers are permissible under specific derogations, for example, if necessary to protect the vital interests of the data subject who is incapable of giving consent (Art. 49(1)(f)). All transfers must fully comply with the GDPR (Recital 101).

### Q7 · Limitation – keyword trap the re-ranker cannot fix
"Data protection officer" matches Arts. 37–39 almost word for word, but the fine is in **Art. 83(4)**, which only says *"Articles 8, 11, 25 to 39"*. The gold chunk is not in the bi-encoder's top 10, so the cross-encoder never sees it.

In [15]:
results['dpo_fine'] = run_both('What fine can a company get for failing to appoint a data protection officer?', gold=['Art. 83(4)'])

### Q: What fine can a company get for failing to appoint a data protection officer?
*Gold chunk(s): Art. 83(4)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,5,Art. 39(1),0.628,-2.38,-4,
2,3,Art. 38(5),0.603,-1.61,-1,
3,2,Art. 38(3),0.601,-0.76,1,
4,4,Art. 38(6),0.596,-1.71,0,
5,1,Art. 37(2),0.593,0.05,4,
6,6,Art. 38(4),0.586,-2.43,0,
7,10,Art. 36(3),0.582,-6.62,-3,
8,9,Art. 83(2) [part 2],0.580,-6.48,-1,
9,8,Art. 83(5),0.580,-3.87,1,
10,7,Art. 37(1),0.577,-3.51,3,


**Baseline top 3:** Art. 39(1), Art. 38(5), Art. 38(3)  
**Re-ranked top 3:** Art. 37(2), Art. 38(3), Art. 38(5)

**Answer 1 (baseline):**
> The provided context does not contain enough information to answer this.

**Answer 2 (re-ranked):**
> The provided context does not contain enough information to answer this.

### Q8 · Counter-example – re-ranking makes it worse
The bi-encoder already ranks the exception list **Art. 17(3)** first; the cross-encoder prefers the general erasure rules (17(1), 17(2)) and pushes it to #4.

In [16]:
results['erasure_exceptions'] = run_both('What are the exceptions to the right to erasure?', gold=['Art. 17(3)'])

### Q: What are the exceptions to the right to erasure?
*Gold chunk(s): Art. 17(3)*

bi_rank,ce_rank,id,bi_score,ce_score,shift,gold
1,4,Art. 17(3),0.690,2.68,-3,★
2,1,Art. 17(2),0.653,3.77,1,
3,3,Recital 66,0.639,3.44,0,
4,2,Art. 17(1),0.635,3.73,2,
5,8,Art. 89(3),0.531,-9.73,-3,
6,6,Recital 65,0.524,-5.86,0,
7,5,Art. 18(2),0.500,-4.58,2,
8,10,Art. 89(2),0.487,-9.84,-2,
9,9,Recital 156,0.484,-9.75,0,
10,7,Art. 20(3),0.475,-7.54,3,


**Baseline top 3:** Art. 17(3), Art. 17(2), Recital 66  
**Re-ranked top 3:** Art. 17(2), Art. 17(1), Recital 66

**Answer 1 (baseline):**
> The right to erasure does not apply if processing is necessary for exercising freedom of expression and information (Art. 17(3)(a)), compliance with a legal obligation or public interest tasks (Art. 17(3)(b)), public interest in public health (Art. 17(3)(c)), archiving, scientific, historical, or statistical purposes where erasure would impair objectives (Art. 17(3)(d)), or for legal claims (Art. 17(3)(e)).

**Answer 2 (re-ranked):**
> The provided context does not contain enough information to answer this.

## 9. Rank-shift summary

In [17]:
rows = []
for key, res in results.items():
    t = res["table"]
    g = t[t.gold == "★"]
    # if several gold chunks were retrieved, report the one the re-ranker moved the most
    best = g.sort_values(["shift", "ce_rank"], ascending=[False, True]).iloc[0] if len(g) else None
    q = next(q for q in QUERIES if q["key"] == key)
    rows.append(dict(
        query=q["label"].split(" · ")[0],
        failure_type=q["label"].split(" · ")[1],
        gold_chunk=best.id if best is not None else "not in top 10",
        baseline_rank=int(best.bi_rank) if best is not None else None,
        reranked_rank=int(best.ce_rank) if best is not None else None,
        gold_in_top3_baseline=bool(best is not None and best.bi_rank <= 3),
        gold_in_top3_reranked=bool(best is not None and best.ce_rank <= 3)))
shift_df = pd.DataFrame(rows).astype({"baseline_rank": "Int64", "reranked_rank": "Int64"})
shift_df["movement"] = [f"#{b} → #{r}" if pd.notna(b) else "not retrieved" for b, r in zip(shift_df.baseline_rank, shift_df.reranked_rank)]
shift_df

,query,failure_type,gold_chunk,baseline_rank,reranked_rank,gold_in_top3_baseline,gold_in_top3_reranked,movement
0,Q1,"Keyword trap / synonym – ""advertising"" vs. ""direct marketing""",Art. 21(3),6,1,False,True,#6 → #1
1,Q2,Exception / corner case – special-category data the person made public,Art. 9(2) [part 2],10,1,False,True,#10 → #1
2,Q3,"Negation – ""not prohibited""",Art. 9(2) [part 3],6,3,False,True,#6 → #3
3,Q4,Exception / role limit – when a controller may refuse a data subject,Art. 12(5),7,3,False,True,#7 → #3
4,Q5,"Exception – ""free of charge … unless""",Art. 12(5),7,2,False,True,#7 → #2
5,Q6,Buried chunk – international transfers without an adequacy decision,Art. 49(1) [part 2],9,3,False,True,#9 → #3
6,Q7,Limitation – keyword trap the re-ranker cannot fix,not in top 10,<NA>,<NA>,False,False,not retrieved
7,Q8,Counter-example – re-ranking makes it worse,Art. 17(3),1,4,True,False,#1 → #4


## 10. Latency profiling

Each stage is timed separately over repeated runs (after a warm-up call) so one-off model loading is excluded.
Retrieval and re-ranking run locally on CPU. LLM time is network + Gemini inference, taken from the generation calls above
(when answers come from `results/llm_cache.json`, the latency recorded on the original call is used).

In [18]:
N_REPEAT = 20
qs = [q["query"] for q in QUERIES]
baseline_retrieve(qs[0]); rerank(qs[0], baseline_retrieve(qs[0]))   # warm-up

t_bi, t_ce = [], []
for _ in range(N_REPEAT):
    for q in qs:
        t0 = time.perf_counter(); c = baseline_retrieve(q); t_bi.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); rerank(q, c);            t_ce.append(time.perf_counter() - t0)

ms = lambda xs: (1000 * statistics.mean(xs), 1000 * statistics.median(xs), 1000 * np.percentile(xs, 95))
# LLM latency: use per-call timings stored in the cache when available. The answers shipped in
# results/llm_cache.json were generated in the original live run, whose per-stage LLM timings
# (8 calls per pipeline, gemini-2.5-flash, free tier) were recorded as mean / median / p95 ms:
ORIGINAL_RUN_LLM_MS = {"Answer 1": (1496.4, 1446.4, 1848.5), "Answer 2": (1353.1, 1387.3, 1622.5)}
llm1 = [r["t_llm1"] for r in results.values() if isinstance(r["t_llm1"], float)]
llm2 = [r["t_llm2"] for r in results.values() if isinstance(r["t_llm2"], float)]
llm_row = lambda name, xs: (f"LLM generation – {name}", *(ms(xs) if len(xs) == len(results)
                            else ORIGINAL_RUN_LLM_MS[name]))

lat = pd.DataFrame([
    ("Bi-encoder: embed query + FAISS top-10 search", *ms(t_bi)),
    ("Cross-encoder: score 10 (query, chunk) pairs", *ms(t_ce)),
    ("Pipeline 1 retrieval total (bi-encoder only)", *ms(t_bi)),
    ("Pipeline 2 retrieval total (bi-encoder + re-rank)", *ms([a + b for a, b in zip(t_bi, t_ce)])),
    llm_row("Answer 1", llm1),
    llm_row("Answer 2", llm2),
], columns=["stage", "mean_ms", "median_ms", "p95_ms"]).round(1)
print(f"{N_REPEAT} repeats x {len(qs)} queries = {len(t_bi)} timings per stage (CPU)")
lat

20 repeats x 8 queries = 160 timings per stage (CPU)


,stage,mean_ms,median_ms,p95_ms
0,Bi-encoder: embed query + FAISS top-10 search,14.3,14.4,16.3
1,"Cross-encoder: score 10 (query, chunk) pairs",183.7,173.9,254.1
2,Pipeline 1 retrieval total (bi-encoder only),14.3,14.4,16.3
3,Pipeline 2 retrieval total (bi-encoder + re-rank),198.0,188.7,264.2
4,LLM generation – Answer 1,1496.4,1446.4,1848.5
5,LLM generation – Answer 2,1353.1,1387.3,1622.5


## 11. Scaling check – why not cross-encode the whole document?

To make the scaling argument concrete, score **every** chunk in the corpus with the cross-encoder for one query
and compare against the bi-encoder search over the same corpus.

In [19]:
q = qs[0]
t0 = time.perf_counter(); baseline_retrieve(q); t_bi_full = time.perf_counter() - t0
t0 = time.perf_counter(); full_scores = cross_encoder.predict([(q, t) for t in texts], batch_size=64); t_ce_full = time.perf_counter() - t0
per_pair = t_ce_full / len(texts)

scale = pd.DataFrame([
    ("Bi-encoder search over all chunks (vectors pre-computed)", len(texts), 1000 * t_bi_full),
    ("Cross-encoder over top-10 only", 10, 1000 * statistics.median(t_ce)),
    ("Cross-encoder over ALL chunks of this one PDF", len(texts), 1000 * t_ce_full),
    ("Cross-encoder, extrapolated: 1,000 such documents", 1000 * len(texts), 1000 * per_pair * 1000 * len(texts)),
    ("Cross-encoder, extrapolated: 1M-chunk collection", 1_000_000, 1000 * per_pair * 1_000_000),
], columns=["setting", "pairs_scored_per_query", "time_ms"])
scale["time_human"] = scale.time_ms.apply(lambda m: f"{m:.0f} ms" if m < 1e4 else (f"{m/1000:.0f} s" if m < 3.6e6 else f"{m/3.6e6:.1f} h"))
scale

,setting,pairs_scored_per_query,time_ms,time_human
0,Bi-encoder search over all chunks (vectors pre-computed),598,1.278333e+01,13 ms
1,Cross-encoder over top-10 only,10,1.739290e+02,174 ms
2,Cross-encoder over ALL chunks of this one PDF,598,6.076513e+03,6077 ms
3,"Cross-encoder, extrapolated: 1,000 such documents",598000,6.076513e+06,1.7 h
4,"Cross-encoder, extrapolated: 1M-chunk collection",1000000,1.016139e+07,2.8 h


## 12. Save results

All answers, rank tables and timings are written to `results/` so they can be referenced in the report.

In [20]:
os.makedirs("results", exist_ok=True)
shift_df.to_csv("results/rank_shift_summary.csv", index=False)
lat.to_csv("results/latency.csv", index=False)
scale.to_csv("results/scaling.csv", index=False)
with open("results/answers.md", "w") as f:
    for key, r in results.items():
        f.write(f"## {r['query']}\n\nGold: {', '.join(r['gold'])}\n\n"
                f"Baseline top 3: {', '.join(r['base_top3'])}\n\nRe-ranked top 3: {', '.join(r['rr_top3'])}\n\n"
                f"{r['table'].to_markdown(index=False)}\n\n"
                f"**Answer 1 (baseline):**\n\n{r['answer1']}\n\n**Answer 2 (re-ranked):**\n\n{r['answer2']}\n\n---\n\n")
print(os.listdir("results"))

['llm_cache.json', 'answers.md', 'scaling.csv', 'latency.csv', 'rank_shift_summary.csv']
